# Trabajo 1 - Econometría II 2026-1S
## Modelación SARIMA de la Tasa de Desocupación Nacional de Colombia

**Integrantes:** [Santiago Vargas, Dilan Tovar, Maria Orjuela]

**Profesores:** Milena Hoyos, Luis Luna, German Rodriguez

**Universidad Nacional de Colombia - Facultad de Ciencias Económicas**

---

### Descripción

Este notebook aplica la metodología Box-Jenkins para modelar la **Tasa de Desocupación (TD)** mensual nacional de Colombia, con datos provenientes de la Gran Encuesta Integrada de Hogares (GEIH) del DANE para el período enero 2001 a marzo 2026 (303 observaciones).

### Estructura del notebook

0. **Configuración** — Importación de librerías y rutas relativas.
1. **Datos** — Lectura del Excel del DANE y generación del CSV limpio.
2. **Análisis exploratorio** — Visualización inicial y descomposición.
3. **Identificación** — FAC, FACP y pruebas de raíz unitaria.
4. **Estimación** — Ajuste de modelos SARIMA candidatos.
5. **Validación** — Diagnóstico de residuos.
6. **Pronóstico** — Proyección a 10 meses.

### Fuente de los datos

DANE — Gran Encuesta Integrada de Hogares (GEIH). Anexo mensual con corte a marzo de 2026.

Archivo original: `data/raw/anex-GEIH-mar2026.xlsx`

## Bloque 0: Configuración

Importación de paquetes y definición de rutas relativas usando `pathlib`. Esto garantiza que el código corra en cualquier computador sin modificar manualmente las rutas.

In [5]:
# %% Importación de paquetes ============================

# Trabajar con rutas relativas en python
from pathlib import Path

# Módulos de numpy, pandas, matplotlib y scipy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import jarque_bera, probplot

# Módulos de statsmodels
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller, kpss

# Configuración estética de matplotlib
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["figure.dpi"] = 100
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["savefig.bbox"] = "tight"
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

print("Paquetes importados correctamente.")

Paquetes importados correctamente.


In [6]:
# %% Definición de rutas relativas =====================

# Detectar la raíz del proyecto:
# Si estamos ejecutando desde notebooks/, subir un nivel para llegar a la raíz.
# Si ya estamos en la raíz, usar el directorio actual.
notebook_path = Path.cwd()
if notebook_path.name == "notebooks":
    PROJECT_ROOT = notebook_path.parent
else:
    PROJECT_ROOT = notebook_path

# Definir carpetas estándar
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
FIGURES = PROJECT_ROOT / "outputs" / "figures"
TABLES = PROJECT_ROOT / "outputs" / "tables"

# Crear las carpetas si no existen (idempotente: no falla si ya existen)
for d in [DATA_PROCESSED, FIGURES, TABLES]:
    d.mkdir(parents=True, exist_ok=True)

# Verificar
print(f"Raíz del proyecto:  {PROJECT_ROOT}")
print(f"Datos crudos:       {DATA_RAW}")
print(f"Datos procesados:   {DATA_PROCESSED}")
print(f"Figuras:            {FIGURES}")
print(f"Tablas:             {TABLES}")

Raíz del proyecto:  c:\postergrupo_mds\poster
Datos crudos:       c:\postergrupo_mds\poster\data\raw
Datos procesados:   c:\postergrupo_mds\poster\data\processed
Figuras:            c:\postergrupo_mds\poster\outputs\figures
Tablas:             c:\postergrupo_mds\poster\outputs\tables


## Bloque 1: ETL — Lectura del Excel y extracción de la serie

El anexo del DANE tiene una estructura compleja:
- La hoja **"Total nacional"** contiene la serie mensual desde 2001.
- Los años están en la fila 12 (columnas 1, 13, 25, ...).
- Los meses están en la fila 13.
- La **Tasa de Desocupación (TD)** está en la fila 17.

Vamos a:
1. Leer la hoja "Total nacional".
2. Extraer la fila de TD.
3. Construir el índice temporal mensual (enero 2001 a marzo 2026).
4. Guardar la serie como CSV limpio en `data/processed/td_nacional.csv`.

In [7]:
# %% Lectura del Excel del DANE ========================

ruta_excel = DATA_RAW / "anex-GEIH-mar2026.xlsx"
print(f"Leyendo: {ruta_excel}")

# Leer la hoja "Total nacional" sin encabezados (estructura irregular)
df_raw = pd.read_excel(
    ruta_excel,
    sheet_name="Total nacional",
    header=None
)

print(f"Forma del DataFrame crudo: {df_raw.shape}")
print(f"  → {df_raw.shape[0]} filas, {df_raw.shape[1]} columnas")

Leyendo: c:\postergrupo_mds\poster\data\raw\anex-GEIH-mar2026.xlsx
Forma del DataFrame crudo: (65, 304)
  → 65 filas, 304 columnas


In [8]:
# %% Extracción de la Tasa de Desocupación =============

# La fila 17 (índice 16) tiene la Tasa de Desocupación
# Verificar que efectivamente es esa fila
nombre_fila = df_raw.iloc[16, 0]
print(f"Nombre de la fila 17: '{nombre_fila}'")

# Extraer la fila completa
fila_td = df_raw.iloc[16]

# Construir la serie mensual:
# - Los años están en la fila 12 en las columnas 1, 13, 25, ...
# - Cada año ocupa 12 columnas consecutivas (Ene a Dic).
# - El primer año es 2001 (columna 1), el último año disponible es 2026.

valores = []
fechas = []

for anio_idx, anio in enumerate(range(2001, 2027)):
    col_inicio = 1 + anio_idx * 12  # columna donde empieza enero del año
    for mes in range(12):
        col = col_inicio + mes
        if col < len(fila_td):
            val = fila_td.iloc[col]
            if pd.notna(val) and val != 0:
                valores.append(float(val))
                fechas.append(pd.Timestamp(year=anio, month=mes + 1, day=1))

# Crear la serie de pandas con índice temporal
td_serie = pd.Series(valores, index=fechas, name="td")

# Asignar nombre al índice
td_serie.index.name = "fecha"

print(f"\nObservaciones extraídas: {len(td_serie)}")
print(f"Rango temporal:          {td_serie.index.min().strftime('%Y-%m')} a {td_serie.index.max().strftime('%Y-%m')}")
print(f"\nPrimeras 5 observaciones:")
print(td_serie.head())
print(f"\nÚltimas 5 observaciones:")
print(td_serie.tail())

Nombre de la fila 17: 'Tasa de Desocupación (TD)'

Observaciones extraídas: 303
Rango temporal:          2001-01 a 2026-03

Primeras 5 observaciones:
fecha
2001-01-01    16.622326
2001-02-01    17.434206
2001-03-01    15.811933
2001-04-01    14.515078
2001-05-01    14.035833
Name: td, dtype: float64

Últimas 5 observaciones:
fecha
2025-11-01     7.024858
2025-12-01     7.977104
2026-01-01    10.859453
2026-02-01     9.234958
2026-03-01     8.793386
Name: td, dtype: float64


In [9]:
# %% Estadísticas descriptivas =========================

print("Estadísticas descriptivas de la Tasa de Desocupación:")
print(td_serie.describe().round(2))

Estadísticas descriptivas de la Tasa de Desocupación:
count    303.00
mean      11.59
std        2.45
min        7.02
25%        9.73
50%       11.18
75%       12.88
max       21.97
Name: td, dtype: float64


In [10]:
# %% Guardado del CSV procesado ========================

ruta_csv = DATA_PROCESSED / "td_nacional.csv"

# Convertir la serie a DataFrame para guardar con dos columnas
td_df = td_serie.reset_index()
td_df.columns = ["fecha", "td"]

# Guardar como CSV
td_df.to_csv(ruta_csv, index=False)

print(f"CSV guardado en: {ruta_csv}")
print(f"Tamaño: {ruta_csv.stat().st_size} bytes")
print(f"\nPrimeras 3 filas del CSV:")
print(td_df.head(3))

CSV guardado en: c:\postergrupo_mds\poster\data\processed\td_nacional.csv
Tamaño: 7056 bytes

Primeras 3 filas del CSV:
       fecha         td
0 2001-01-01  16.622326
1 2001-02-01  17.434206
2 2001-03-01  15.811933
